In [8]:
# Throw a ball at 100 different velocities.

import jax
import mujoco
from mujoco import mjx
import numpy as np

XML=r"""
<mujoco>
  <worldbody>
    <body>
      <freejoint/>
      <geom size=".15" mass="1" type="sphere"/>
    </body>
  </worldbody>
</mujoco>
"""

model = mujoco.MjModel.from_xml_string(XML)
mjx_model = mjx.put_model(model)

@jax.vmap
def batched_step(vel):
  mjx_data = mjx.make_data(mjx_model)
  qvel = mjx_data.qvel.at[0].set(vel)
  mjx_data = mjx_data.replace(qvel=qvel)
  pos = mjx.step(mjx_model, mjx_data).qpos[0]
  return pos

# 对比直接Mujoco
def run_mujoco(vel):
  for v in vel:
    data = mujoco.MjData(model)
    data.qvel[0] = v
    mujoco.mj_step(model, data)
  
# mjx
vel = jax.numpy.arange(0.0, 5.0, 0.01)
pos = jax.jit(batched_step)(vel)
print(pos)

vel_np = np.arange(0.0, 5.0, 0.01)

# mujoco
run_mujoco(vel_np)

[0.00000000e+00 2.00000013e-05 4.00000026e-05 6.00000021e-05
 8.00000053e-05 9.99999975e-05 1.20000004e-04 1.40000004e-04
 1.60000011e-04 1.80000003e-04 1.99999995e-04 2.20000016e-04
 2.40000008e-04 2.60000001e-04 2.80000007e-04 2.99999985e-04
 3.20000021e-04 3.40000028e-04 3.60000005e-04 3.80000012e-04
 3.99999990e-04 4.19999997e-04 4.40000033e-04 4.60000010e-04
 4.80000017e-04 5.00000024e-04 5.20000001e-04 5.39999979e-04
 5.60000015e-04 5.79999993e-04 5.99999970e-04 6.20000006e-04
 6.40000042e-04 6.60000020e-04 6.80000056e-04 7.00000033e-04
 7.20000011e-04 7.40000047e-04 7.60000024e-04 7.80000002e-04
 7.99999980e-04 8.20000016e-04 8.39999993e-04 8.59999971e-04
 8.80000065e-04 9.00000043e-04 9.20000020e-04 9.40000056e-04
 9.60000034e-04 9.80000012e-04 1.00000005e-03 1.02000008e-03
 1.04000000e-03 1.06000004e-03 1.07999996e-03 1.10000011e-03
 1.12000003e-03 1.14000007e-03 1.15999999e-03 1.18000002e-03
 1.19999994e-03 1.22000009e-03 1.24000001e-03 1.26000005e-03
 1.28000008e-03 1.300000

In [ ]:
%timeit run_mujoco(vel_np)
%timeit jax.jit(batched_step)(vel)

197 ms ± 12.9 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
